<a href="https://colab.research.google.com/github/parkmiseong/Git/blob/main/CowMuzzle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

소프트웨어공학 소의 코지문 인식 시스템 구현

In [1]:
#캐글 설치
!pip install kaggle

In [2]:
#토큰 파일 복사 -> kaggle.json 파일 / kaggle 세팅에서 API 토큰 생성
'''
from google.colab import files
files.upload()
'''

'\nfrom google.colab import files\nfiles.upload()\n'

In [3]:
#프로젝트 폴더에 json 파일 추가 권한 할당
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [4]:
#캐글의 연구와 학습 목적으로 사용이 가능한 데이터셋 다운로드
!kaggle datasets download sharifashik/cow-muzzle-dataset

Dataset URL: https://www.kaggle.com/datasets/sharifashik/cow-muzzle-dataset
License(s): apache-2.0
100% 67.9M/67.9M [00:04<00:00, 14.7MB/s]



In [5]:
#cow-muzzle-dataset 압축 해제
from zipfile import ZipFile
file_name = 'cow-muzzle-dataset.zip'
with ZipFile(file_name, 'r') as zip:
  zip.extractall('/content/cow-muzzle-dataset')
  print('done')

done


코드 작성

In [6]:

##https://github.com/dileepajay/HOMBENAI/blob/main/script_extract/node_to_print.py 참고하기
import cv2
import os
from ultralytics import YOLO

# 1. 딥러닝 기반 YOLOv8 모델 로드 (cascade.xml 대체)
model_path = './CowMuzzle_Project/v1_train/weights/best.pt' # 학습된 모델 가중치
yolo_model = YOLO(model_path)

# 2. 이미지 불러오기 (흑백 변환 불필요)
image_path = 'test_images/cow_face.jpg'
img = cv2.imread(image_path)

# 3. 비문(코) 영역 탐지 (AI 예측 수행)
# conf=0.5: 50% 이상 비문이라고 확신하는 경우에만 결과 반환
results = yolo_model.predict(source=img, conf=0.5, verbose=False)

# 4. 탐지된 영역 크롭 및 저장
output_dir = 'noses_yolo/'
os.makedirs(output_dir, exist_ok=True)

# 탐지된 바운딩 박스(네모 상자)들을 순회
boxes = results[0].boxes
for i, box in enumerate(boxes):
    # YOLO는 좌표를 [시작X, 시작Y, 끝X, 끝Y] 형태로 반환합니다.
    x1, y1, x2, y2 = map(int, box.xyxy[0])

    # 원본 컬러 이미지에서 코 부분만 정확히 자르기 (Crop)
    cropped_nose = img[y1:y2, x1:x2]

    # 잘라낸 이미지 저장
    output_path = os.path.join(output_dir, f'extracted_nose_{i}.jpg')
    cv2.imwrite(output_path, cropped_nose)

print(f"총 {len(boxes)}개의 비문을 성공적으로 추출하여 저장했습니다.")